<a href="https://colab.research.google.com/github/swrobuts/dav/blob/main/notebooks/13_Fallstudie_HR_Analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fallstudie: HR Analytics – Mitarbeiterabgang vorhersagen

**Szenario**: Du bist HR-Analyst bei einem mittelständischen Unternehmen. Die Geschäftsleitung ist besorgt über eine steigende Fluktuation und beauftragt dich, die Daten auszuwerten. Ziel: Herausfinden, welche Faktoren zum Mitarbeiterabgang führen, und Hochrisiko-Profile identifizieren.

**Vollständiger DAV-Workflow:**
1. Business Understanding
2. Daten laden & inspizieren
3. Datenqualität prüfen
4. Explorative Analyse (EDA)
5. Feature Engineering
6. Risikoprofil erstellen
7. Erkenntnisse & Handlungsempfehlungen

---

In [ ]:
# ── Setup: Pakete installieren (nur in Colab nötig) ──────────────────────────
import subprocess, sys
packages = ['ydata-profiling', 'missingno', 'plotly', 'kaleido']
for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)
print("Setup abgeschlossen!")

In [ ]:
# ── Daten laden von GitHub ────────────────────────────────────────────────────
BASE_URL = "https://raw.githubusercontent.com/swrobuts/dav/main/data/"
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
print("Bibliotheken geladen!")

## Phase 1: Business Understanding

**Kernfragen:**

1. Wie hoch ist die Abgang-Rate insgesamt und nach Abteilung?
2. Welche Faktoren korrelieren am stärksten mit dem Abgang?
3. Wie hängen Gehalt, Überstunden und Zufriedenheit zusammen?
4. Wer sind die Hochrisiko-Mitarbeiter?

**Warum ist das wichtig?**
- Die Neubesetzung einer Stelle kostet durchschnittlich 6-9 Monatsgehälter
- Frühzeitige Erkennung ermöglicht gezielte Maßnahmen (Gehaltserhöhung, Mentoring, Aufgabenwechsel)
- Datenbasis: 1.500 Mitarbeiterdatensätze

In [ ]:
# ── Daten laden ───────────────────────────────────────────────────────────────
df = pd.read_csv(BASE_URL + "mitarbeiter_hr.csv")
print(f"Datensatz geladen: {df.shape[0]:,} Mitarbeiter, {df.shape[1]} Merkmale")
print(f"Abgang-Rate gesamt: {df['abgang'].mean()*100:.1f}%")
print(f"Abgegangene Mitarbeiter: {df['abgang'].sum()} von {len(df)}")
print()
df.head(10)

In [ ]:
# ── Erste Inspektion ──────────────────────────────────────────────────────────
print("=== Datentypen ===")
print(df.dtypes)
print()
print("=== Statistische Kennzahlen ===")
df.describe().round(2)

In [ ]:
# ── Fehlende Werte & Datenqualität ────────────────────────────────────────────
print("=== Fehlende Werte ===")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "Keine fehlenden Werte!")
print()
print(f"Duplikate: {df.duplicated().sum()}")
print()
print("=== Kategoriale Spalten ===")
for col in ['abteilung', 'geschlecht', 'zufriedenheit']:
    print(f"  {col}: {df[col].value_counts().to_dict()}")
print()
print(f"Altersverteilung: {df['alter'].min()} – {df['alter'].max()} Jahre (Median: {df['alter'].median():.0f})")
print(f"Gehaltsrange: {df['gehalt'].min():,} – {df['gehalt'].max():,} EUR")

In [ ]:
# ── Abgang-Rate nach Abteilung ────────────────────────────────────────────────
abgang_abt = (df.groupby('abteilung')
               .agg(gesamt=('abgang', 'count'),
                    abgegangen=('abgang', 'sum'))
               .reset_index())
abgang_abt['abgang_rate'] = (abgang_abt['abgegangen'] / abgang_abt['gesamt'] * 100).round(1)
abgang_abt = abgang_abt.sort_values('abgang_rate', ascending=False)

fig = make_subplots(rows=1, cols=2, subplot_titles=['Abgang-Rate nach Abteilung (%)', 'Absolute Abgänge'])

fig.add_trace(
    go.Bar(x=abgang_abt['abteilung'], y=abgang_abt['abgang_rate'],
           marker_color='salmon', text=abgang_abt['abgang_rate'],
           texttemplate='%{text}%', textposition='outside', name='Rate'),
    row=1, col=1
)
fig.add_trace(
    go.Bar(x=abgang_abt['abteilung'], y=abgang_abt['abgegangen'],
           marker_color='steelblue', text=abgang_abt['abgegangen'],
           textposition='outside', name='Anzahl'),
    row=1, col=2
)
fig.update_layout(title='Mitarbeiterabgang nach Abteilung', height=450, showlegend=False)
fig.show()

print(abgang_abt.to_string(index=False))

In [ ]:
# ── Korrelationsanalyse: Was beeinflusst den Abgang? ─────────────────────────
num_cols = ['gehalt', 'ueberstunden_pro_woche', 'zufriedenheit',
            'betriebszugehoerigkeit_jahre', 'befoerderungen_letzte_3j', 'alter', 'abgang']
corr = df[num_cols].corr().round(2)

fig = go.Figure(data=go.Heatmap(
    z=corr.values,
    x=corr.columns.tolist(),
    y=corr.columns.tolist(),
    colorscale='RdBu',
    zmin=-1, zmax=1,
    text=corr.values,
    texttemplate='%{text}',
))
fig.update_layout(title='Korrelationsmatrix – Fokus auf Abgang', height=500)
fig.show()

# Stärkste Korrelationen mit Abgang
corr_abgang = corr['abgang'].drop('abgang').sort_values(key=abs, ascending=False)
print("Stärkste Korrelationen mit Abgang:")
print(corr_abgang.to_string())

In [ ]:
# ── Gehalt vs. Zufriedenheit vs. Abgang ──────────────────────────────────────
df_plot = df.copy()
df_plot['abgang_label'] = df_plot['abgang'].map({0: 'Geblieben', 1: 'Abgegangen'})

fig = px.scatter(
    df_plot,
    x='gehalt',
    y='zufriedenheit',
    color='abgang_label',
    facet_col='abteilung',
    facet_col_wrap=4,
    title='Gehalt vs. Zufriedenheit nach Abgang-Status und Abteilung',
    labels={'gehalt': 'Gehalt (EUR)', 'zufriedenheit': 'Zufriedenheit (1-5)'},
    color_discrete_map={'Geblieben': 'steelblue', 'Abgegangen': 'salmon'},
    opacity=0.6,
    height=500,
)
fig.show()

In [ ]:
# ── Überstunden-Analyse nach Abgang-Status ────────────────────────────────────
df_plot['abgang_label'] = df_plot['abgang'].map({0: 'Geblieben', 1: 'Abgegangen'})

fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Überstunden nach Abgang-Status (Box)', 'Ø Überstunden nach Abteilung'])

for status, color in [('Geblieben', 'steelblue'), ('Abgegangen', 'salmon')]:
    subset = df_plot[df_plot['abgang_label'] == status]['ueberstunden_pro_woche']
    fig.add_trace(
        go.Box(y=subset, name=status, marker_color=color, boxmean=True),
        row=1, col=1
    )

ueber_abt = df.groupby(['abteilung', 'abgang'])['ueberstunden_pro_woche'].mean().reset_index()
ueber_abt['abgang_label'] = ueber_abt['abgang'].map({0: 'Geblieben', 1: 'Abgegangen'})
for status, color in [('Geblieben', 'steelblue'), ('Abgegangen', 'salmon')]:
    subset = ueber_abt[ueber_abt['abgang_label'] == status]
    fig.add_trace(
        go.Bar(x=subset['abteilung'], y=subset['ueberstunden_pro_woche'],
               name=status, marker_color=color, showlegend=False),
        row=1, col=2
    )

fig.update_layout(title='Überstunden-Analyse', height=450)
fig.update_yaxes(title_text='Überstunden/Woche')
fig.show()

In [ ]:
# ── Encoding kategoriale Spalten ──────────────────────────────────────────────
df_enc = df.copy()

# One-Hot-Encoding für Abteilung
abt_dummies = pd.get_dummies(df_enc['abteilung'], prefix='abt')
df_enc = pd.concat([df_enc, abt_dummies], axis=1)

# Ordinal-Encoding für Geschlecht
df_enc['geschlecht_enc'] = df_enc['geschlecht'].map({'M': 0, 'W': 1, 'D': 2})

print("Encoding abgeschlossen!")
print(f"Neue Abteilungs-Dummy-Spalten: {list(abt_dummies.columns)}")
print(f"Gesamte Spaltenanzahl: {df_enc.shape[1]}")
print()
print(df_enc[['mitarbeiter_id', 'abteilung'] + list(abt_dummies.columns)].head(5))

In [ ]:
# ── Feature Engineering: Gehaltsklasse & Risiko-Score ────────────────────────
# Gehaltsklasse
df_enc['gehalt_klasse'] = pd.cut(
    df_enc['gehalt'],
    bins=[0, 40000, 60000, 95001],
    labels=['niedrig', 'mittel', 'hoch']
)

# Risiko-Score (normalisiert, 0-100)
# Höheres Risiko bei: niedrige Zufriedenheit, hohe Überstunden, kein Wachstum
df_enc['risiko_score'] = (
    (5 - df_enc['zufriedenheit']) / 4 * 40          # Zufriedenheit (40 Punkte)
    + df_enc['ueberstunden_pro_woche'] / 15 * 30    # Überstunden (30 Punkte)
    + (df_enc['befoerderungen_letzte_3j'] == 0).astype(int) * 20  # Keine Beförderung (20 Punkte)
    + (df_enc['betriebszugehoerigkeit_jahre'] < 2).astype(int) * 10  # Kurze Zugehörigkeit (10 Punkte)
).round(1)

print("Feature Engineering abgeschlossen!")
print(f"Gehaltsklassen: {df_enc['gehalt_klasse'].value_counts().to_dict()}")
print(f"Risiko-Score Statistik:")
print(df_enc['risiko_score'].describe().round(1))

In [ ]:
# ── Hochrisiko-Mitarbeiter identifizieren ────────────────────────────────────
hochrisiko = df_enc[df_enc['risiko_score'] >= df_enc['risiko_score'].quantile(0.80)].copy()
hochrisiko = hochrisiko.sort_values('risiko_score', ascending=False)

print(f"Hochrisiko-Mitarbeiter (Top 20%): {len(hochrisiko)}")
print(f"Tatsächliche Abgang-Rate in dieser Gruppe: {hochrisiko['abgang'].mean()*100:.1f}%")
print(f"Abgang-Rate in Niedrigrisiko-Gruppe: {df_enc[df_enc['risiko_score'] < df_enc['risiko_score'].quantile(0.80)]['abgang'].mean()*100:.1f}%")
print()
print("Top 15 Hochrisiko-Mitarbeiter:")
cols_show = ['mitarbeiter_id', 'abteilung', 'gehalt', 'ueberstunden_pro_woche',
             'zufriedenheit', 'befoerderungen_letzte_3j', 'risiko_score', 'abgang']
print(hochrisiko[cols_show].head(15).to_string(index=False))

# Visualisierung
fig = px.scatter(
    df_enc,
    x='risiko_score',
    y='zufriedenheit',
    color=df_enc['abgang'].map({0: 'Geblieben', 1: 'Abgegangen'}),
    size='ueberstunden_pro_woche',
    hover_data=['mitarbeiter_id', 'abteilung', 'gehalt'],
    title='Risiko-Score vs. Zufriedenheit (Blasengröße = Überstunden)',
    labels={'risiko_score': 'Risiko-Score', 'zufriedenheit': 'Zufriedenheit (1-5)'},
    color_discrete_map={'Geblieben': 'steelblue', 'Abgegangen': 'salmon'},
    opacity=0.65,
    height=500,
)
fig.show()

## Zusammenfassung & Handlungsempfehlungen

**Wichtigste Erkenntnisse:**

1. **Zufriedenheit ist der stärkste Prädiktor**: Mitarbeiter mit Zufriedenheit 1-2 zeigen deutlich höhere Abgangsraten.
2. **Überstunden verstärken das Risiko**: Abteilungen mit hohem Überstundenpensum (Vertrieb, Produktion) haben höhere Fluktuation.
3. **Keine Beförderung = Risikofaktor**: Mitarbeiter ohne Karriereentwicklung in 3 Jahren sind gefährdeter.
4. **Frühe Phase kritisch**: Niedrige Betriebszugehörigkeit (<2 Jahre) ist ein Warnsignal.

**Handlungsempfehlungen:**
- Regelmäßige Mitarbeitergespräche für Hochrisiko-Mitarbeiter einführen
- Überstunden-Management in Vertrieb und Produktion prüfen
- Transparente Karrierepfade kommunizieren
- Onboarding-Programm für neue Mitarbeiter verbessern

---
*DAV Lab – Prof. Dr. Robert Butscher – THWS Business School*